# IRIS: 01 Prepare data

Turns the raw rainfall, disease and boundary files into the small files the dashboard and the later notebooks read.

| Output (in `data/processed/`) | What it holds |
| --- | --- |
| `dekad_rain.csv` | 10-day rainfall per province since 2015 with each province's own 90th percentile; Metro Manila's row drives the rainfall alert |
| `province_weekly_p90.csv` | Each province's 90th percentile of weekly rain, used to score the live forecast |
| `weekly_national.csv` | The weekly national rain signal joined to weekly leptospirosis cases, for the lag analysis |
| `provinces.geojson` | Simplified province shapes keyed by `PCODE`, used in notebook 04 to place one forecast point per province (optional step, needs the boundary geodatabase) |

**Why all provinces, for a Metro Manila tool?** Leptospirosis case counts are only open at the national level, so
the rain signal that is matched to them has to be national too: the share of provinces with an extreme week.

**Province units.** The rainfall file uses 81 units with codes like `PH01028`. NCR is one unit (`PH13074`).

Run top to bottom. On Colab, run the setup cell first and point `RAW` at the uploaded `datasets` folder.

In [1]:
# Setup. On Colab, uncomment the Drive lines and set RAW to the datasets folder.
# from google.colab import drive
# drive.mount('/content/drive')
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')   # project folder
RAW = ROOT / 'datasets'          # on Colab, e.g. Path('/content/drive/MyDrive/IRIS/datasets')
OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

RAIN_CSV = RAW / 'phl-rainfall-subnat-full.csv'
DISEASE_XLSX = RAW / 'Diseases Data (for NBL).xlsx'
BOUNDARY_GDB = RAW / 'phl_admin_boundaries.gdb'   # folder, or the downloaded .gdb.zip

BASELINE = (1981, 2024)   # full years used for percentiles and thresholds
TRIGGER_FROM = '2015-01-01'

## 1. Rainfall

Keep the province rows (`adm_level == 2`). `rfh` is the 10-day (dekad) rainfall total in mm and `r1h` is the
rolling 1-month total, which stands in for the 30-day total in the proposal. The last two dekads are marked
`prelim` by the publisher; they are kept for the trigger but left out of the baseline.

In [2]:
rain = pd.read_csv(RAIN_CSV, parse_dates=['date'])
rain = rain[rain['adm_level'] == 2][['date', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg', 'r1h', 'version']].copy()
UNITS = sorted(rain['PCODE'].unique())
print(len(UNITS), 'province units;', rain['date'].min().date(), 'to', rain['date'].max().date())
base = rain[rain['date'].dt.year.between(*BASELINE) & (rain['version'] == 'final')]

81 province units; 1981-01-01 to 2026-10-11


## 2. Trigger table for the dashboard

Each province's own 90th percentile of 10-day rainfall (baseline years) is the alert threshold. The table keeps
dekads from 2015 on so the app stays small.

In [3]:
p90 = base.groupby('PCODE')['rfh'].quantile(0.90).rename('p90_mm')
trig = rain[rain['date'] >= TRIGGER_FROM].join(p90, on='PCODE')
trig['above_p90'] = trig['rfh'] > trig['p90_mm']
trig = trig[['date', 'PCODE', 'rfh', 'rfh_avg', 'p90_mm', 'above_p90', 'version']].round({'rfh': 2, 'rfh_avg': 2, 'p90_mm': 2})
trig.to_csv(OUT / 'dekad_rain.csv', index=False)
print(len(trig), 'rows; latest dekad', trig['date'].max().date(),
      '| provinces above p90 then:', int(trig.loc[trig['date'] == trig['date'].max(), 'above_p90'].sum()))

34425 rows; latest dekad 2026-10-11 | provinces above p90 then: 2


## 3. Weekly national series for the lag analysis

Each dekad total is spread evenly over its days (dekad 3 runs 8 to 11 days), summed into ISO weeks per
province, and then summarized nationally: the share of provinces above their own weekly 90th percentile and the
mean weekly rainfall. Weekly leptospirosis counts (2019 to 2023 in this file) are reshaped to year and week and
joined on ISO year and week, which approximates DOH morbidity weeks. Missing weeks stay missing.

In [4]:
r = rain[['date', 'PCODE', 'rfh']].copy()
nxt = r['date'] + pd.offsets.MonthBegin(1)
r['end'] = np.where(r['date'].dt.day == 21, nxt, r['date'] + pd.Timedelta(days=10))
r['ndays'] = (pd.to_datetime(r['end']) - r['date']).dt.days
daily = r.loc[r.index.repeat(r['ndays'])].copy()
daily['day'] = daily['date'] + pd.to_timedelta(daily.groupby(level=0).cumcount(), unit='D')
daily['mm'] = daily['rfh'] / daily['ndays']
iso = daily['day'].dt.isocalendar()
daily['iso_year'], daily['iso_week'] = iso['year'].astype(int), iso['week'].astype(int)

wk = daily.groupby(['PCODE', 'iso_year', 'iso_week'])['mm'].agg(['sum', 'size']).reset_index()
wk = wk[wk['size'] == 7]                                   # drop partial weeks at the ends
wk_p90 = wk[wk['iso_year'].between(*BASELINE)].groupby('PCODE')['sum'].quantile(0.90)
wk_p90.rename('weekly_p90_mm').round(2).to_csv(OUT / 'province_weekly_p90.csv')   # used by the live forecast
wk['above'] = wk['sum'] > wk['PCODE'].map(wk_p90)
nat = wk.groupby(['iso_year', 'iso_week']).agg(
    share_prov_above_p90=('above', 'mean'), mean_rain_mm=('sum', 'mean')).reset_index()

dis = pd.read_excel(DISEASE_XLSX)
lep = [c for c in dis.columns if c.lower().startswith('leptospirosis')]
lepto = (dis[['Week'] + lep].dropna(subset=['Week'])
         .melt(id_vars='Week', var_name='col', value_name='cases'))
lepto['iso_year'] = lepto['col'].str[-4:].astype(int)
lepto['iso_week'] = lepto['Week'].astype(int)
lepto = lepto[['iso_year', 'iso_week', 'cases']]

weekly = nat.merge(lepto, on=['iso_year', 'iso_week'], how='left')
weekly = weekly[weekly['iso_year'] >= lepto['iso_year'].min() - 1]
weekly['week_start'] = pd.to_datetime(
    weekly['iso_year'].astype(str) + weekly['iso_week'].astype(str).str.zfill(2) + '1', format='%G%V%u')
weekly['covid_period'] = weekly['iso_year'].isin([2020, 2021])
weekly.round({'share_prov_above_p90': 4, 'mean_rain_mm': 2}).to_csv(OUT / 'weekly_national.csv', index=False)
print(weekly['cases'].notna().sum(), 'weeks with case counts;',
      weekly.groupby('iso_year')['cases'].count().to_dict())

227 weeks with case counts; {2018: 0, 2019: 52, 2020: 28, 2021: 52, 2022: 52, 2023: 43, 2024: 0, 2025: 0, 2026: 0}


## 4. Province shapes (optional)

Skip this if `provinces.geojson` is already in `data/processed/`. It reads only the province layer of the
boundary geodatabase, merges the 2025 units into the 81 rainfall units and simplifies the shapes to about 1 MB. Notebook 04 uses them
to place one rainfall forecast point in each province.
Needs `geopandas` (preinstalled on Colab).

In [5]:
RUN_GEO = not (OUT / 'provinces.geojson').exists()
if RUN_GEO:
    import geopandas as gpd
    shp = gpd.read_file(BOUNDARY_GDB, layer='phl_admin2', columns=['adm2_pcode'])
    MERGE = {'PH13039': 'PH13074', 'PH13075': 'PH13074', 'PH13076': 'PH13074', 'PH09097': 'PH19007',
             'PH11086': 'PH11024', 'PH19088': 'PH19087', 'PH19099': 'PH19087'}
    shp['PCODE'] = shp['adm2_pcode'].replace(MERGE)
    assert set(shp['PCODE']) == set(UNITS)
    shp = shp.dissolve('PCODE').reset_index()[['PCODE', 'geometry']].to_crs(4326)
    shp['geometry'] = shp.simplify(0.01, preserve_topology=True)
    shp.to_file(OUT / 'provinces.geojson', driver='GeoJSON')
print(sorted(p.name for p in OUT.iterdir()))

['dekad_rain.csv', 'disease_lag.csv', 'event_study.csv', 'forecast_backtest.csv', 'forecast_coefs.csv', 'forecast_scores.csv', 'lag_curve.png', 'lag_results.csv', 'ncr_barangays.geojson', 'ncr_flood_events.csv', 'ncr_month_outlook.csv', 'province_points.csv', 'province_weekly_p90.csv', 'provinces.geojson', 'weekly_case_baseline.csv', 'weekly_national.csv']
